# Audio + Flow Tensor Motion Anomaly Inference

音声モデルは従来通り推論し、動きモデルは学習artifactに保存されたflow系特徴の3x3時間tensorで推論します。音声・動き・同期スコアを統合し、推論結果を表で表示します。


## 処理の流れ

1. `INFERENCE_RUN_MODE='single'` では従来通り、1つのartifactで指定targetを推論します。
2. `INFERENCE_RUN_MODE='split_validation'` では分割検証で学習した各モデルを使い、ordered pairごとの推論結果をCSVとMarkdownで出力します。
3. 同一モデルに対する異常データ推論は1回だけ実行し、各pair結果へ再利用します。


In [ ]:
from __future__ import annotations

import importlib
import json
import sys
from pathlib import Path
from typing import Any

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        fallback_iterable = [] if iterable is None else iterable
        return fallback_iterable


def _find_import_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'forklift_features').exists():
            return candidate
    raise FileNotFoundError(f'Could not find src/forklift_features from {start}')


PROJECT_ROOT = _find_import_root()
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from forklift_features import flow_tensor, pipeline as feature_pipeline, scoring, split_validation
from forklift_features.paths import find_project_root, safe_path_part

flow_tensor = importlib.reload(flow_tensor)
feature_pipeline = importlib.reload(feature_pipeline)
scoring = importlib.reload(scoring)
split_validation = importlib.reload(split_validation)

pd.set_option('display.max_columns', 160)


In [ ]:
PROJECT_ROOT = find_project_root()
MOVIE_PREPROCESS_DIR = PROJECT_ROOT / 'data' / 'movie_preprocess'
BASE_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'anomaly_feature_poc'
OUTPUT_DIR = BASE_OUTPUT_DIR
FEATURE_CACHE_DIR = OUTPUT_DIR / 'sample_feature_cache'
MODEL_PATH = OUTPUT_DIR / 'isolation_forest_feature_poc.joblib'
TARGET_OUTPUT_ROOT = OUTPUT_DIR / 'target_inference'
TARGET_COMPARISON_DIR = TARGET_OUTPUT_ROOT / '_comparison'

INFERENCE_RUN_MODE = 'split_validation'  # 'single' or 'split_validation'
SPLIT_GROUP_COUNT = 2
SPLIT_RANDOM_STATE = 42
SPLIT_RUN_NAME = f'{SPLIT_GROUP_COUNT}groups_seed{SPLIT_RANDOM_STATE}'
SPLIT_OUTPUT_DIR = BASE_OUTPUT_DIR / 'split_validation' / SPLIT_RUN_NAME
SPLIT_FEATURE_CACHE_DIR = SPLIT_OUTPUT_DIR / 'sample_feature_cache'
SPLIT_DATA_DIR = PROJECT_ROOT / 'data' / 'validation_splits' / SPLIT_RUN_NAME
SPLIT_MANIFEST_PATH = SPLIT_OUTPUT_DIR / 'split_manifest.csv'
SPLIT_DATA_MANIFEST_PATH = SPLIT_DATA_DIR / 'split_manifest.csv'
REUSE_SPLIT_INFERENCE_CACHE = True

INFERENCE_TARGET_SAMPLE_LIST_PATH = PROJECT_ROOT / 'data' / 'inference_target_sample_list.csv'
INFERENCE_TARGET_SAMPLE_LIST_LEGACY_PATH = PROJECT_ROOT / 'data' / 'inference_target_sample_list'
if not INFERENCE_TARGET_SAMPLE_LIST_PATH.exists() and INFERENCE_TARGET_SAMPLE_LIST_LEGACY_PATH.exists():
    INFERENCE_TARGET_SAMPLE_LIST_PATH = INFERENCE_TARGET_SAMPLE_LIST_LEGACY_PATH
TARGET_SELECTION_MODE = 'sample_list' if INFERENCE_TARGET_SAMPLE_LIST_PATH.exists() else 'random_untrained'
TARGET_SPECS: list[dict[str, Any]] = []
TARGET_RANDOM_COUNT = 8
TARGET_RANDOM_STATE = 42
TOP_N_ANOMALIES = 8
WRITE_TARGET_RAW_FLOW = False

RUN_AUDIO_EVENT_DEPTH_ESTIMATION = True
AUDIO_EVENT_DEPTH_MODEL_NAME = 'depth-anything/Depth-Anything-V2-Small-hf'
AUDIO_EVENT_DEPTH_SCORE_THRESHOLD = 0.70
AUDIO_EVENT_DEPTH_MIN_SEPARATION_SEC = 0.5
AUDIO_EVENT_DEPTH_MAX_EVENTS_PER_SAMPLE: int | None = None
AUDIO_EVENT_DEPTH_VIEWS: tuple[str, ...] | None = None  # Noneなら学習artifactで有効なfront/rearを使う
AUDIO_EVENT_DEPTH_CMAP = 'magma_r'
AUDIO_EVENT_DEPTH_PERCENTILE_RANGE = (2.0, 98.0)
AUDIO_EVENT_DEPTH_OVERLAY_ALPHA = 0.45
AUDIO_EVENT_DEPTH_OUTPUT_SUBDIR = 'audio_event_depth'
AUDIO_EVENT_DEPTH_OVERWRITE = False

print(f'INFERENCE_RUN_MODE: {INFERENCE_RUN_MODE}')
print(f'MODEL_PATH: {MODEL_PATH}')
print(f'target selection mode: {TARGET_SELECTION_MODE}')
print(f'SPLIT_RUN_NAME: {SPLIT_RUN_NAME}')


In [ ]:
# ------------------------------------------------------------
# セル概要: 音声イベント時刻の単眼深度推定
# ------------------------------------------------------------
# - scored_df の audio_event_score が閾値以上の時刻を音声イベント代表点として選びます。
# - front/rear の該当フレームへ forklift_monocular_depth_estimation_poc.ipynb と同じ深度推定を行います。
# - 推定結果は sampleごとの audio_event_depth ディレクトリへ比較画像として保存します。
# ------------------------------------------------------------

_AUDIO_EVENT_DEPTH_ESTIMATOR: Any | None = None


def load_audio_event_depth_estimator(model_name: str = AUDIO_EVENT_DEPTH_MODEL_NAME):
    try:
        import torch
        from transformers import pipeline
    except ImportError as exc:
        raise ImportError(
            'Depth estimation requires transformers and torch. '
            'Run `%pip install transformers torch pillow opencv-python matplotlib pandas numpy`.'
        ) from exc

    device = 0 if torch.cuda.is_available() else -1
    print(f'loading audio-event depth model: {model_name} (device={device})')
    return pipeline('depth-estimation', model=model_name, device=device)


def get_audio_event_depth_estimator():
    global _AUDIO_EVENT_DEPTH_ESTIMATOR
    if _AUDIO_EVENT_DEPTH_ESTIMATOR is None:
        _AUDIO_EVENT_DEPTH_ESTIMATOR = load_audio_event_depth_estimator()
    return _AUDIO_EVENT_DEPTH_ESTIMATOR


def depth_result_to_array(depth_result: dict[str, Any], target_size: tuple[int, int]) -> np.ndarray:
    target_width, target_height = target_size
    if 'predicted_depth' in depth_result:
        predicted = depth_result['predicted_depth']
        if hasattr(predicted, 'detach'):
            depth = predicted.detach().cpu().numpy()
        else:
            depth = np.asarray(predicted)
        depth = np.squeeze(depth).astype(np.float32)
    elif 'depth' in depth_result:
        depth = np.asarray(depth_result['depth'], dtype=np.float32)
    else:
        raise KeyError(f'unsupported depth-estimation result keys: {sorted(depth_result.keys())}')

    if depth.shape[:2] != (target_height, target_width):
        depth = cv2.resize(depth, (target_width, target_height), interpolation=cv2.INTER_CUBIC)
    return depth.astype(np.float32)


def normalize_depth_for_display(
    depth: np.ndarray,
    percentile_range: tuple[float, float] = AUDIO_EVENT_DEPTH_PERCENTILE_RANGE,
) -> np.ndarray:
    finite = depth[np.isfinite(depth)]
    if finite.size == 0:
        return np.zeros(depth.shape, dtype=np.float32)

    lower, upper = np.percentile(finite, percentile_range)
    if not np.isfinite(lower) or not np.isfinite(upper) or upper <= lower:
        lower, upper = float(np.nanmin(finite)), float(np.nanmax(finite))
    if upper <= lower:
        return np.zeros(depth.shape, dtype=np.float32)

    normalized = (depth - lower) / (upper - lower)
    return np.clip(normalized, 0.0, 1.0).astype(np.float32)


def colorize_depth(depth: np.ndarray, cmap_name: str = AUDIO_EVENT_DEPTH_CMAP) -> np.ndarray:
    cmap = plt.get_cmap(cmap_name)
    return (cmap(normalize_depth_for_display(depth))[..., :3] * 255).astype(np.uint8)


def blend_depth_overlay(
    frame_rgb: np.ndarray,
    depth_color_rgb: np.ndarray,
    alpha: float = AUDIO_EVENT_DEPTH_OVERLAY_ALPHA,
) -> np.ndarray:
    blended = frame_rgb.astype(np.float32) * (1.0 - alpha) + depth_color_rgb.astype(np.float32) * alpha
    return np.clip(blended, 0, 255).astype(np.uint8)


def estimate_depth_for_frame(depth_estimator: Any, frame_rgb: np.ndarray) -> dict[str, np.ndarray]:
    image = Image.fromarray(frame_rgb)
    depth_result = depth_estimator(image)
    depth = depth_result_to_array(depth_result, image.size)
    depth_color = colorize_depth(depth)
    overlay = blend_depth_overlay(frame_rgb, depth_color)
    return {'depth': depth, 'depth_color': depth_color, 'overlay': overlay}


def sample_view_video_path(sample: pd.Series, view: str) -> Path | None:
    value = sample.get(f'{view}_path')
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass

    path = Path(str(value))
    candidates = [path] if path.is_absolute() else [PROJECT_ROOT / path, MOVIE_PREPROCESS_DIR / path]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return path.resolve() if path.exists() else None


def read_rgb_frame_at_time(video_path: Path, time_sec: float) -> tuple[np.ndarray, dict[str, Any]]:
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f'Could not open video: {video_path}')

    try:
        fps = float(capture.get(cv2.CAP_PROP_FPS) or 0.0)
        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
        if fps <= 0:
            raise RuntimeError(f'Could not read FPS from video: {video_path}')

        requested_frame_index = int(round(float(time_sec) * fps))
        frame_index = min(requested_frame_index, max(frame_count - 1, 0)) if frame_count > 0 else requested_frame_index
        capture.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
        ok, frame_bgr = capture.read()
        if not ok or frame_bgr is None:
            raise RuntimeError(f'Could not read frame {frame_index} from {video_path}')
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        actual_frame_index = int(capture.get(cv2.CAP_PROP_POS_FRAMES) or (frame_index + 1)) - 1
        return frame_rgb, {
            'fps': fps,
            'frame_count': frame_count,
            'duration_sec': frame_count / fps if frame_count > 0 else np.nan,
            'frame_index': actual_frame_index,
            'width': width,
            'height': height,
        }
    finally:
        capture.release()


def save_depth_comparison_image(
    *,
    output_path: Path,
    frame_rgb: np.ndarray,
    depth_color: np.ndarray,
    overlay: np.ndarray,
    title: str,
) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
    fig.suptitle(title, fontsize=12)
    for ax, image, label in zip(axes, [frame_rgb, depth_color, overlay], ['frame', 'relative depth', 'overlay']):
        ax.imshow(image)
        ax.set_title(label)
        ax.axis('off')
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def select_audio_event_rows(scored_df: pd.DataFrame) -> pd.DataFrame:
    if scored_df is None or scored_df.empty or 'audio_event_score' not in scored_df.columns or 'time' not in scored_df.columns:
        return pd.DataFrame()

    work = scored_df.copy()
    work['_time_sec'] = pd.to_numeric(work['time'], errors='coerce')
    work['_audio_event_score'] = pd.to_numeric(work['audio_event_score'], errors='coerce')
    work = work.dropna(subset=['_time_sec', '_audio_event_score'])
    work = work[work['_audio_event_score'] >= float(AUDIO_EVENT_DEPTH_SCORE_THRESHOLD)]
    if work.empty:
        return pd.DataFrame()

    selected_rows: list[dict[str, Any]] = []
    selected_times: list[float] = []
    ranked = work.sort_values('_audio_event_score', ascending=False).reset_index(drop=True)
    min_sep = max(0.0, float(AUDIO_EVENT_DEPTH_MIN_SEPARATION_SEC))
    max_events = AUDIO_EVENT_DEPTH_MAX_EVENTS_PER_SAMPLE
    for rank, (_, row) in enumerate(ranked.iterrows(), start=1):
        time_sec = float(row['_time_sec'])
        if min_sep > 0.0 and any(abs(time_sec - prev) < min_sep for prev in selected_times):
            continue
        payload = row.to_dict()
        payload['audio_event_rank'] = int(rank)
        selected_rows.append(payload)
        selected_times.append(time_sec)
        if max_events is not None and len(selected_rows) >= int(max_events):
            break

    selected_df = pd.DataFrame(selected_rows)
    if selected_df.empty:
        return selected_df
    selected_df = selected_df.sort_values('_time_sec').reset_index(drop=True)
    selected_df['audio_event_index'] = np.arange(1, len(selected_df) + 1)
    return selected_df


def depth_views_for_context(model_context: dict[str, Any]) -> tuple[str, ...]:
    if AUDIO_EVENT_DEPTH_VIEWS is not None:
        return tuple(str(view) for view in AUDIO_EVENT_DEPTH_VIEWS)
    return tuple(str(view) for view in model_context.get('motion_cameras', ('front', 'rear')))


def audio_event_depth_output_path(sample_output_dir: Path, view: str, event_row: pd.Series, frame_index: int) -> Path:
    time_part = safe_path_part(f"t{float(event_row['_time_sec']):08.3f}s")
    rank_part = f"event{int(event_row['audio_event_index']):02d}"
    score_part = safe_path_part(f"score{float(event_row['_audio_event_score']):.3f}")
    return sample_output_dir / AUDIO_EVENT_DEPTH_OUTPUT_SUBDIR / safe_path_part(view) / f'{rank_part}_{time_part}_{safe_path_part(view)}_frame{int(frame_index):06d}_{score_part}.png'


def write_audio_event_depth_for_sample(
    *,
    sample: pd.Series,
    sample_scored_df: pd.DataFrame,
    model_context: dict[str, Any],
    sample_output_dir: Path,
) -> list[dict[str, Any]]:
    event_rows = select_audio_event_rows(sample_scored_df)
    if event_rows.empty:
        return []

    rows: list[dict[str, Any]] = []
    depth_estimator = None
    sample_id = str(sample.get('sample_id', 'unknown'))
    target_group = str(sample.get('target_group', ''))
    for _, event_row in event_rows.iterrows():
        time_sec = float(event_row['_time_sec'])
        for view in depth_views_for_context(model_context):
            video_path = sample_view_video_path(sample, view)
            if video_path is None:
                continue
            frame_rgb, frame_meta = read_rgb_frame_at_time(video_path, time_sec)
            output_path = audio_event_depth_output_path(sample_output_dir, view, event_row, int(frame_meta['frame_index']))
            if AUDIO_EVENT_DEPTH_OVERWRITE or not output_path.exists():
                if depth_estimator is None:
                    depth_estimator = get_audio_event_depth_estimator()
                depth_payload = estimate_depth_for_frame(depth_estimator, frame_rgb)
                title = (
                    f"sample={sample_id} view={view} t={time_sec:.3f}s "
                    f"audio_event_score={float(event_row['_audio_event_score']):.3f}"
                )
                save_depth_comparison_image(
                    output_path=output_path,
                    frame_rgb=frame_rgb,
                    depth_color=depth_payload['depth_color'],
                    overlay=depth_payload['overlay'],
                    title=title,
                )
            rows.append({
                'sample_id': sample_id,
                'category': sample.get('category', sample.get('target_category', '')),
                'environment': sample.get('environment', sample.get('target_environment', '')),
                'target_group': target_group,
                'view': view,
                'event_index': int(event_row['audio_event_index']),
                'event_rank': int(event_row['audio_event_rank']),
                'time_sec': time_sec,
                'audio_event_score': float(event_row['_audio_event_score']),
                'audio_anomaly_score': float(event_row.get('audio_anomaly_score', np.nan)),
                'frame_index': int(frame_meta['frame_index']),
                'fps': float(frame_meta['fps']),
                'video_path': str(video_path),
                'output_path': str(output_path),
            })
    return rows


def scored_rows_for_sample(scored_df: pd.DataFrame, sample_id: str) -> pd.DataFrame:
    if scored_df is None or scored_df.empty:
        return pd.DataFrame()
    if 'video_id' in scored_df.columns:
        return scored_df[scored_df['video_id'].astype(str).eq(str(sample_id))].copy()
    if 'sample_id' in scored_df.columns:
        return scored_df[scored_df['sample_id'].astype(str).eq(str(sample_id))].copy()
    return pd.DataFrame()


def write_audio_event_depth_outputs(
    *,
    target_samples: pd.DataFrame,
    all_target_scored_df: pd.DataFrame,
    model_context: dict[str, Any],
    target_output_root: Path,
    comparison_dir: Path,
) -> pd.DataFrame:
    comparison_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = comparison_dir / 'audio_event_depth_manifest.csv'
    if not RUN_AUDIO_EVENT_DEPTH_ESTIMATION:
        empty_df = pd.DataFrame()
        empty_df.to_csv(manifest_path, index=False)
        return empty_df

    manifest_rows: list[dict[str, Any]] = []
    for record in tqdm(target_samples.to_dict('records'), total=len(target_samples), desc='audio event depth'):
        sample = pd.Series(record)
        sample_id = str(sample.get('sample_id', 'unknown'))
        sample_scored_df = scored_rows_for_sample(all_target_scored_df, sample_id)
        sample_output_dir = target_output_root / safe_path_part(sample_id)
        manifest_rows.extend(write_audio_event_depth_for_sample(
            sample=sample,
            sample_scored_df=sample_scored_df,
            model_context=model_context,
            sample_output_dir=sample_output_dir,
        ))

    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(manifest_path, index=False)
    print(f'saved audio-event depth outputs: {manifest_path} ({len(manifest_df)} images)')
    return manifest_df


In [ ]:
MOVIE_PREPROCESS_MANIFEST_PATH = MOVIE_PREPROCESS_DIR / 'movie_preprocess_manifest.csv'
TRAIN_SAMPLE_LIST_PATH = PROJECT_ROOT / 'data' / 'train_sample_list.csv'


def path_or_none(value: Any) -> Path | None:
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if not text or text.lower() == 'nan':
        return None
    return Path(text)


def coerce_path_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ['front_path', 'rear_path', 'audio_path', 'front_frame_map_path', 'rear_frame_map_path']:
        if col in out.columns:
            out[col] = out[col].apply(path_or_none)
    return out


def infer_sample_id_from_preprocess_path(path_value: Any, suffix: str = '_front.mp4') -> str:
    name = Path(str(path_value)).name
    if name.endswith(suffix):
        return name[:-len(suffix)]
    return Path(name).stem


def build_preprocess_paths(sample_id: str, category: str, environment: str) -> dict[str, Path]:
    target_dir = MOVIE_PREPROCESS_DIR / str(category) / str(environment)
    front_path = target_dir / f'{sample_id}_front.mp4'
    rear_path = target_dir / f'{sample_id}_rear.mp4'
    return {
        'front_path': front_path,
        'rear_path': rear_path,
        'audio_path': target_dir / f'{sample_id}_audio.wav',
        'front_frame_map_path': front_path.with_name(f'{front_path.stem}_frame_map.csv'),
        'rear_frame_map_path': rear_path.with_name(f'{rear_path.stem}_frame_map.csv'),
    }


def scan_movie_preprocess_inventory() -> pd.DataFrame:
    rows = []
    for front_path in sorted(MOVIE_PREPROCESS_DIR.glob('*/*/*_front.mp4')):
        sample_id = front_path.name[:-len('_front.mp4')]
        category = front_path.parent.parent.name
        environment = front_path.parent.name
        paths = build_preprocess_paths(sample_id, category, environment)
        if not paths['rear_path'].exists():
            continue
        rows.append({
            'sample_id': sample_id,
            'category': category,
            'environment': environment,
            **paths,
            'plot_label': sample_id,
            'status': 'scanned_from_files',
            'audio_status': 'exists' if paths['audio_path'].exists() else 'missing',
        })
    return pd.DataFrame(rows)


def load_movie_preprocess_inventory(manifest_path: Path = MOVIE_PREPROCESS_MANIFEST_PATH) -> pd.DataFrame:
    if not manifest_path.exists():
        return scan_movie_preprocess_inventory()
    manifest_df = pd.read_csv(manifest_path)
    rows = []
    for _, row in manifest_df.iterrows():
        category = str(row.get('category', 'unknown'))
        environment = str(row.get('environment', 'unknown'))
        sample_id = str(row.get('sample_id', '')).strip()
        if not sample_id or sample_id.lower() == 'nan':
            sample_id = infer_sample_id_from_preprocess_path(row.get('front_output_path', row.get('input_video_path', '')))
        if not sample_id or sample_id.lower() == 'nan':
            continue
        paths = build_preprocess_paths(sample_id, category, environment)
        if not paths['front_path'].exists() or not paths['rear_path'].exists():
            continue
        rows.append({
            'sample_id': sample_id,
            'category': category,
            'environment': environment,
            **paths,
            'plot_label': sample_id,
            'status': row.get('status', 'manifest'),
            'audio_status': row.get('audio_status', 'exists' if paths['audio_path'].exists() else 'missing'),
        })
    return pd.DataFrame(rows).drop_duplicates(['sample_id', 'category', 'environment']).reset_index(drop=True)


def load_train_sample_ids(path: Path = TRAIN_SAMPLE_LIST_PATH, artifact_train_sample_ids: set[str] | None = None) -> set[str]:
    ids = set(artifact_train_sample_ids or set())
    if path.exists():
        train_df = pd.read_csv(path)
        if 'sample_id' in train_df.columns:
            ids.update(train_df['sample_id'].astype(str).tolist())
    return ids


def select_random_untrained_targets(inventory_df: pd.DataFrame, train_sample_ids: set[str]) -> pd.DataFrame:
    candidates = inventory_df[~inventory_df['sample_id'].astype(str).isin(train_sample_ids)].copy()
    if candidates.empty:
        candidates = inventory_df.copy()
    count = len(candidates) if int(TARGET_RANDOM_COUNT) < 0 else min(int(TARGET_RANDOM_COUNT), len(candidates))
    return candidates.sample(n=count, random_state=TARGET_RANDOM_STATE).sort_values(['category', 'environment', 'sample_id']).reset_index(drop=True)


def rows_from_target_specs(target_list_df: pd.DataFrame, inventory_df: pd.DataFrame) -> pd.DataFrame:
    if 'sample_id' not in target_list_df.columns:
        raise ValueError('target sample list must contain a sample_id column')
    rows = []
    for _, spec in target_list_df.iterrows():
        if {'front_path', 'rear_path'}.issubset(target_list_df.columns) and pd.notna(spec.get('front_path')) and pd.notna(spec.get('rear_path')):
            front_path = Path(str(spec.get('front_path')))
            rear_path = Path(str(spec.get('rear_path')))
            audio_value = spec.get('audio_path') if 'audio_path' in target_list_df.columns else None
            audio_path = Path(str(audio_value)) if pd.notna(audio_value) else front_path.with_name(f"{spec.get('sample_id')}_audio.wav")
            front_frame_map_value = spec.get('front_frame_map_path') if 'front_frame_map_path' in target_list_df.columns else None
            rear_frame_map_value = spec.get('rear_frame_map_path') if 'rear_frame_map_path' in target_list_df.columns else None
            front_frame_map_path = Path(str(front_frame_map_value)) if pd.notna(front_frame_map_value) else front_path.with_name(f'{front_path.stem}_frame_map.csv')
            rear_frame_map_path = Path(str(rear_frame_map_value)) if pd.notna(rear_frame_map_value) else rear_path.with_name(f'{rear_path.stem}_frame_map.csv')
            rows.append({
                'sample_id': str(spec.get('sample_id')),
                'category': str(spec.get('category', 'unknown')),
                'environment': str(spec.get('environment', 'unknown')),
                'front_path': front_path,
                'rear_path': rear_path,
                'audio_path': audio_path if audio_path.exists() else None,
                'front_frame_map_path': front_frame_map_path if front_frame_map_path.exists() else None,
                'rear_frame_map_path': rear_frame_map_path if rear_frame_map_path.exists() else None,
                'plot_label': str(spec.get('plot_label', spec.get('sample_id'))),
            })
            continue

        part = inventory_df[inventory_df['sample_id'].astype(str).eq(str(spec['sample_id']))].copy()
        if 'category' in target_list_df.columns and pd.notna(spec.get('category')):
            part = part[part['category'].astype(str).eq(str(spec['category']))]
        if 'environment' in target_list_df.columns and pd.notna(spec.get('environment')):
            part = part[part['environment'].astype(str).eq(str(spec['environment']))]
        if part.empty:
            raise ValueError(f'inference target not found in movie_preprocess inventory: {dict(spec)}')
        row = part.iloc[0].copy()
        if 'plot_label' in target_list_df.columns and pd.notna(spec.get('plot_label')):
            row['plot_label'] = spec.get('plot_label')
        rows.append(row)
    return pd.DataFrame(rows).reset_index(drop=True)


def load_inference_target_sample_list(path: Path, inventory_df: pd.DataFrame) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'inference target sample list was not found: {path}')
    return rows_from_target_specs(pd.read_csv(path), inventory_df)


def select_single_target_samples(inventory_df: pd.DataFrame, train_sample_ids: set[str]) -> pd.DataFrame:
    if TARGET_SELECTION_MODE == 'sample_list':
        selected = load_inference_target_sample_list(INFERENCE_TARGET_SAMPLE_LIST_PATH, inventory_df)
    elif TARGET_SELECTION_MODE == 'manual':
        selected = rows_from_target_specs(pd.DataFrame(TARGET_SPECS), inventory_df)
    elif TARGET_SELECTION_MODE == 'random_untrained':
        selected = select_random_untrained_targets(inventory_df, train_sample_ids)
    else:
        raise ValueError("TARGET_SELECTION_MODE must be 'sample_list', 'manual', or 'random_untrained'")
    if selected.empty:
        raise ValueError('No target samples were selected')
    return coerce_path_columns(selected.reset_index(drop=True))


def load_split_manifest(path: Path = SPLIT_MANIFEST_PATH) -> pd.DataFrame:
    manifest_path = path
    if not manifest_path.exists() and SPLIT_DATA_MANIFEST_PATH.exists():
        manifest_path = SPLIT_DATA_MANIFEST_PATH
    if not manifest_path.exists():
        raise FileNotFoundError(f'split manifest was not found: {path} or {SPLIT_DATA_MANIFEST_PATH}')
    manifest_df = pd.read_csv(manifest_path)
    required = {'sample_id', 'category', 'environment', 'split_group', 'split_group_index', 'front_path', 'rear_path'}
    missing = sorted(required - set(manifest_df.columns))
    if missing:
        raise ValueError(f'split manifest is missing columns: {missing}')
    return coerce_path_columns(manifest_df.reset_index(drop=True))


def split_group_order(manifest_df: pd.DataFrame) -> list[str]:
    order_df = (
        manifest_df[['split_group', 'split_group_index']]
        .drop_duplicates()
        .sort_values('split_group_index')
    )
    return order_df['split_group'].astype(str).tolist()


In [ ]:
def load_model_context(model_path: Path, feature_cache_dir: Path) -> dict[str, Any]:
    if not model_path.exists():
        raise FileNotFoundError(f'Trained model was not found: {model_path}')
    artifact = joblib.load(model_path)
    if 'flow_tensor_model' not in artifact or 'score_models' not in artifact or 'audio' not in artifact['score_models']:
        raise ValueError('This inference notebook requires an audio_flow_tensor_event_sync_v4 artifact. Re-run the training notebook.')
    if artifact.get('model_version') != 'audio_flow_tensor_event_sync_v4':
        raise ValueError('This inference notebook requires an audio_flow_tensor_event_sync_v4 artifact. Re-run the training notebook.')

    settings = artifact.get('settings', {})
    audio_sr = int(settings.get('audio_sr', 16000))
    n_mels = int(settings.get('n_mels', 16))
    window_sec = float(settings.get('window_sec', 0.2))
    flow_sample_sec = float(settings.get('flow_sample_sec', 0.1))
    flow_grid = tuple(settings.get('flow_grid', (3, 3)))
    use_front = bool(settings.get('use_front', True))
    use_rear = bool(settings.get('use_rear', True))
    frame_resize_width = settings.get('frame_resize_width', 480)
    flow_analysis_scale = float(settings.get('flow_analysis_scale', 0.75))
    flow_reliable_error_threshold_px = float(settings.get('flow_reliable_error_threshold_px', 1.0))
    flow_tensor_window_sec = float(settings.get('flow_tensor_window_sec', 1.0))
    flow_tensor_hop_sec = float(settings.get('flow_tensor_hop_sec', 0.2))
    flow_tensor_score_aggregation = str(settings.get('flow_tensor_score_aggregation', 'max'))
    model_input_dtype = np.dtype(settings.get('model_input_dtype', 'float32')).type
    flow_cache_version = str(settings.get('flow_cache_version', 'sample_flow_cache_v13'))
    flow_cache_method = 'farneback_event_update_interval_apportioned_0p1s'
    artifact_flow_cache_settings = settings.get('flow_cache_settings', {})
    artifact_flow_method = str(artifact_flow_cache_settings.get('flow_method', flow_cache_method)) if isinstance(artifact_flow_cache_settings, dict) else flow_cache_method
    if flow_cache_version in {'sample_flow_cache_v1', 'sample_flow_cache_v2', 'sample_flow_cache_v3', 'sample_flow_cache_v4', 'sample_flow_cache_v5', 'sample_flow_cache_v6'} or artifact_flow_method != flow_cache_method:
        raise ValueError('This artifact was trained with an old optical-flow cache. Re-run the training notebook.')

    audio_model_artifact = artifact['score_models']['audio']
    motion_model = artifact['flow_tensor_model']
    motion_feature_names = flow_tensor.normalize_motion_feature_names(settings.get('motion_feature_names', motion_model.get('channels', ['flow_x', 'flow_y'])))
    flow_tensor_broad_vib_score_config = settings.get('flow_tensor_broad_vib_score_config', {})
    flow_tensor_reliable_soft_gate_config = settings.get('flow_tensor_reliable_soft_gate_config', flow_tensor.DEFAULT_RELIABLE_SOFT_GATE_CONFIG)
    flow_tensor_accel_impact_score_config = settings.get('flow_tensor_accel_impact_score_config', flow_tensor.DEFAULT_ACCEL_IMPACT_SCORE_CONFIG)
    if len(motion_feature_names) != int(np.asarray(motion_model.get('mean_tensor')).shape[-1]):
        raise ValueError('Motion feature count does not match the trained tensor model. Re-run the training notebook.')
    sync_score_config = artifact.get('sync_score_config', {
        'audio_anomaly_column': 'audio_anomaly_score',
        'motion_anomaly_column': 'motion_anomaly_score',
        'audio_event_column': 'audio_event_score',
        'motion_event_column': 'motion_event_score',
        'max_lag_windows': 2,
    })
    final_score_weights = artifact.get('final_score_weights', {'audio_anomaly_score': 0.45, 'motion_anomaly_score': 0.35, 'sync_score': 0.20})
    plot_score_columns = artifact.get('plot_score_columns', ['anomaly_score', 'audio_anomaly_score', 'motion_anomaly_score', 'sync_score'])
    event_score_calibrations = artifact.get('event_score_calibrations')
    if not isinstance(event_score_calibrations, dict) or 'audio' not in event_score_calibrations or 'motion' not in event_score_calibrations:
        raise ValueError('This artifact does not contain event_score_calibrations. Re-run the training notebook.')
    motion_event_relative_score_config = event_score_calibrations.get(
        'motion_relative',
        settings.get('motion_event_relative_score_config', scoring.DEFAULT_MOTION_EVENT_RELATIVE_SCORE_CONFIG),
    )
    if not isinstance(motion_event_relative_score_config, dict):
        motion_event_relative_score_config = dict(scoring.DEFAULT_MOTION_EVENT_RELATIVE_SCORE_CONFIG)
    train_sample_ids = set(map(str, artifact.get('train_sample_ids', [])))
    validation_split = artifact.get('validation_split', {'enabled': False})

    flow_cache_settings = feature_pipeline.build_flow_cache_settings(
        cache_version=flow_cache_version,
        use_front=use_front,
        use_rear=use_rear,
        flow_sample_sec=flow_sample_sec,
        frame_resize_width=frame_resize_width,
        flow_analysis_scale=flow_analysis_scale,
        flow_grid=flow_grid,
        flow_reliable_error_threshold_px=flow_reliable_error_threshold_px,
        flow_method=flow_cache_method,
    )
    raw_flow_pipeline_kwargs = {
        'flow_cache_settings': flow_cache_settings,
        'cache_dir': feature_cache_dir,
        'use_front': use_front,
        'use_rear': use_rear,
        'flow_sample_sec': flow_sample_sec,
        'frame_resize_width': frame_resize_width,
        'flow_analysis_scale': flow_analysis_scale,
        'flow_grid': flow_grid,
        'flow_reliable_error_threshold_px': flow_reliable_error_threshold_px,
        'enable_cache': True,
    }
    audio_pipeline_kwargs = {
        'audio_sr': audio_sr,
        'window_sec': window_sec,
        'hop_sec': flow_tensor_hop_sec,
        'n_mels': n_mels,
        'label_default': None,
    }

    return {
        'model_path': model_path,
        'artifact': artifact,
        'audio_model_artifact': audio_model_artifact,
        'motion_model': motion_model,
        'motion_feature_names': motion_feature_names,
        'motion_cameras': feature_pipeline.enabled_cameras(use_front=use_front, use_rear=use_rear),
        'raw_flow_pipeline_kwargs': raw_flow_pipeline_kwargs,
        'audio_pipeline_kwargs': audio_pipeline_kwargs,
        'flow_sample_sec': flow_sample_sec,
        'flow_grid': flow_grid,
        'flow_tensor_window_sec': flow_tensor_window_sec,
        'flow_tensor_hop_sec': flow_tensor_hop_sec,
        'flow_tensor_score_aggregation': flow_tensor_score_aggregation,
        'model_input_dtype': model_input_dtype,
        'flow_tensor_broad_vib_score_config': flow_tensor_broad_vib_score_config,
        'flow_tensor_reliable_soft_gate_config': flow_tensor_reliable_soft_gate_config,
        'flow_tensor_accel_impact_score_config': flow_tensor_accel_impact_score_config,
        'sync_score_config': sync_score_config,
        'final_score_weights': final_score_weights,
        'plot_score_columns': plot_score_columns,
        'event_score_calibrations': event_score_calibrations,
        'motion_event_relative_score_config': motion_event_relative_score_config,
        'train_sample_ids': train_sample_ids,
        'validation_split': validation_split,
    }


def build_inference_result_table(target_video_summary: pd.DataFrame) -> pd.DataFrame:
    if target_video_summary is None or target_video_summary.empty:
        return pd.DataFrame()
    motion_attribution_display_cols = [
        'motion_top_camera',
        'motion_top_grid_channel',
        'motion_top_grid_label',
    ]
    inference_base_cols = [
        'video_id', 'target_category', 'target_environment', 'target_group', 'max_anomaly_window_start_sec',
        'max_anomaly_score', 'sync_score_at_max_anomaly',
        'audio_anomaly_score_at_max_anomaly', 'motion_anomaly_score_at_max_anomaly',
    ]
    table = (
        target_video_summary[[c for c in [*inference_base_cols, *motion_attribution_display_cols] if c in target_video_summary.columns]]
        .rename(columns={
            'video_id': 'ファイル名',
            'target_category': '正解ラベル',
            'target_environment': '環境',
            'target_group': '推論データ群',
            'max_anomaly_window_start_sec': '最大異常窓開始時刻',
            'max_anomaly_score': '異常スコア',
            'sync_score_at_max_anomaly': '同期イベントスコア',
            'audio_anomaly_score_at_max_anomaly': '音声異常スコア',
            'motion_anomaly_score_at_max_anomaly': '動き異常スコア',
            'motion_top_camera': '動き異常カメラ',
            'motion_top_grid_channel': '動き異常成分',
            'motion_top_grid_label': '動き異常グリッド',
        })
        .reset_index(drop=True)
    )
    if '異常スコア' in table.columns:
        table = table.sort_values('異常スコア', ascending=False).reset_index(drop=True)
    return table


def summarize_scored_outputs(all_target_scored_df: pd.DataFrame, model_context: dict[str, Any]) -> dict[str, pd.DataFrame]:
    motion_attribution_cols = [
        'motion_top_camera',
        'motion_top_channel',
        'motion_top_grid_row',
        'motion_top_grid_col',
        'motion_top_grid_label',
        'motion_top_grid_channel',
        'motion_flow_x_contribution',
        'motion_flow_y_contribution',
        'motion_t_flow_x_contribution',
        'motion_t_flow_y_contribution',
        'motion_flow_x_broad_vib_score_contribution',
        'motion_flow_y_broad_vib_score_contribution',
        'motion_t_flow_x_broad_vib_score_contribution',
        'motion_t_flow_y_broad_vib_score_contribution',
        'motion_t_accel_impact_x_score_contribution',
        'motion_t_accel_impact_y_score_contribution',
        'motion_top_grid_contribution',
    ]
    if all_target_scored_df is None or all_target_scored_df.empty:
        return {
            'target_video_summary': pd.DataFrame(),
            'inference_result_table': pd.DataFrame(),
            'target_top_windows': pd.DataFrame(),
            'all_target_key_metrics_df': pd.DataFrame(),
        }

    scored_for_summary = all_target_scored_df.copy()
    if 'window_start_sec' not in scored_for_summary.columns:
        scored_for_summary['window_start_sec'] = np.nan
    summary_time = pd.to_numeric(scored_for_summary['time'], errors='coerce').fillna(0.0) if 'time' in scored_for_summary.columns else pd.Series(0.0, index=scored_for_summary.index)
    fallback_start_sec = summary_time - 0.5 * float(model_context['flow_tensor_window_sec'])
    scored_for_summary['window_start_sec'] = pd.to_numeric(scored_for_summary['window_start_sec'], errors='coerce').fillna(fallback_start_sec).clip(lower=0.0)
    if 'window_end_sec' not in scored_for_summary.columns:
        scored_for_summary['window_end_sec'] = scored_for_summary['window_start_sec'] + float(model_context['flow_tensor_window_sec'])
    if 'window_center_sec' not in scored_for_summary.columns:
        scored_for_summary['window_center_sec'] = 0.5 * (scored_for_summary['window_start_sec'] + pd.to_numeric(scored_for_summary['window_end_sec'], errors='coerce').fillna(scored_for_summary['window_start_sec']))
    if 'target_group' not in scored_for_summary.columns:
        scored_for_summary['target_group'] = ''

    best_window_indices = scored_for_summary.groupby('video_id')['anomaly_score'].idxmax()
    best_windows_df = scored_for_summary.loc[best_window_indices].copy()
    best_windows_df = best_windows_df.rename(columns={
        'window_start_sec': 'max_anomaly_window_start_sec',
        'window_end_sec': 'max_anomaly_window_end_sec',
        'window_center_sec': 'max_anomaly_window_center_sec',
        'anomaly_score': 'max_anomaly_score',
        'audio_anomaly_score': 'audio_anomaly_score_at_max_anomaly',
        'motion_anomaly_score': 'motion_anomaly_score_at_max_anomaly',
        'sync_score': 'sync_score_at_max_anomaly',
        'final_anomaly_score': 'final_anomaly_score_at_max_anomaly',
    })
    summary_extra_df = (
        scored_for_summary
        .groupby('video_id', as_index=False)
        .agg(
            top5_mean_anomaly_score=('anomaly_score', scoring.topk_mean),
            p95_anomaly_score=('anomaly_score', lambda s: float(s.quantile(0.95))),
            n_windows=('anomaly_score', 'size'),
        )
    )
    summary_base_cols = [
        'video_id', 'target_category', 'target_environment', 'target_label', 'target_group',
        'max_anomaly_window_start_sec', 'max_anomaly_window_end_sec', 'max_anomaly_window_center_sec',
        'max_anomaly_score', 'sync_score_at_max_anomaly',
        'audio_anomaly_score_at_max_anomaly', 'motion_anomaly_score_at_max_anomaly',
        'final_anomaly_score_at_max_anomaly',
    ]
    target_video_summary = (
        best_windows_df[[c for c in [*summary_base_cols, *motion_attribution_cols] if c in best_windows_df.columns]]
        .merge(summary_extra_df, on='video_id', how='left')
        .sort_values('max_anomaly_score', ascending=False)
        .reset_index(drop=True)
    )
    inference_result_table = build_inference_result_table(target_video_summary)
    window_metric_cols = [
        'video_id', 'target_category', 'target_environment', 'target_label', 'target_group',
        'window_start_sec', 'time', *model_context['plot_score_columns'], 'final_anomaly_score',
        'anomaly_score_smooth', *motion_attribution_cols,
    ]
    target_top_windows = pd.concat(
        [g.nlargest(TOP_N_ANOMALIES, 'anomaly_score') for _, g in scored_for_summary.groupby('video_id', sort=False)],
        ignore_index=True,
    )[[c for c in window_metric_cols if c in scored_for_summary.columns]]
    all_target_key_metrics_df = scored_for_summary[[c for c in window_metric_cols if c in scored_for_summary.columns]].copy()
    return {
        'target_video_summary': target_video_summary,
        'inference_result_table': inference_result_table,
        'target_top_windows': target_top_windows,
        'all_target_key_metrics_df': all_target_key_metrics_df,
    }


def inference_cache_metadata(target_samples: pd.DataFrame, model_context: dict[str, Any]) -> dict[str, Any]:
    model_path = Path(model_context['model_path'])
    stat = model_path.stat()
    metadata = {
        'model_path': str(model_path.resolve()),
        'model_size': int(stat.st_size),
        'model_mtime_ns': int(stat.st_mtime_ns),
        'sample_ids': target_samples['sample_id'].astype(str).tolist() if 'sample_id' in target_samples.columns else [],
        'target_groups': target_samples.get('target_group', pd.Series('', index=target_samples.index)).astype(str).tolist(),
    }
    return metadata


def read_scored_outputs_if_available(comparison_dir: Path, expected_metadata: dict[str, Any] | None = None) -> dict[str, pd.DataFrame] | None:
    paths = {
        'all_target_scored_df': comparison_dir / 'all_target_window_scores.csv',
        'target_video_summary': comparison_dir / 'target_video_scores.csv',
        'inference_result_table': comparison_dir / 'target_inference_result_table.csv',
        'target_top_windows': comparison_dir / 'target_top_windows.csv',
        'all_target_key_metrics_df': comparison_dir / 'all_target_key_metrics.csv',
        'cache_manifest_df': comparison_dir / 'flow_cache_manifest.csv',
    }
    if not all(path.exists() for path in paths.values()):
        return None
    metadata_path = comparison_dir / 'inference_cache_metadata.json'
    if expected_metadata is not None:
        if not metadata_path.exists():
            return None
        try:
            cached_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        except Exception:
            return None
        if cached_metadata != expected_metadata:
            return None
    return {key: pd.read_csv(path) for key, path in paths.items()}


def score_target_samples(
    target_samples: pd.DataFrame,
    *,
    model_context: dict[str, Any],
    target_output_root: Path,
    comparison_dir: Path,
    reuse_cache: bool = False,
    write_raw_flow: bool = WRITE_TARGET_RAW_FLOW,
) -> dict[str, pd.DataFrame]:
    if target_samples.empty:
        raise ValueError('No target samples were selected')
    expected_cache_metadata = inference_cache_metadata(target_samples, model_context)
    if reuse_cache:
        cached = read_scored_outputs_if_available(comparison_dir, expected_cache_metadata)
        if cached is not None:
            cached['audio_event_depth_manifest_df'] = write_audio_event_depth_outputs(
                target_samples=target_samples,
                all_target_scored_df=cached.get('all_target_scored_df', pd.DataFrame()),
                model_context=model_context,
                target_output_root=target_output_root,
                comparison_dir=comparison_dir,
            )
            print(f'reused inference cache: {comparison_dir}')
            return cached

    target_output_root.mkdir(parents=True, exist_ok=True)
    comparison_dir.mkdir(parents=True, exist_ok=True)

    all_feature_parts: list[pd.DataFrame] = []
    all_raw_flow_parts: list[pd.DataFrame] = []
    all_scored_parts: list[pd.DataFrame] = []
    cache_rows: list[dict[str, Any]] = []

    for record in tqdm(target_samples.to_dict('records'), total=len(target_samples), desc=f'score {target_output_root.name}'):
        sample = pd.Series(record)
        sample_id = str(sample.get('sample_id', 'unknown'))
        target_group = str(sample.get('target_group', ''))
        sample_output_dir = target_output_root / safe_path_part(sample_id)
        sample_output_dir.mkdir(parents=True, exist_ok=True)

        raw_flow_df, cache_info = feature_pipeline.load_or_extract_raw_flow(sample, **model_context['raw_flow_pipeline_kwargs'])
        cache_rows.append(cache_info)
        if len(raw_flow_df):
            raw_flow_df = raw_flow_df.copy()
            raw_flow_df['target_group'] = target_group
            all_raw_flow_parts.append(raw_flow_df)
            if write_raw_flow:
                raw_flow_df.to_csv(sample_output_dir / f'{safe_path_part(sample_id)}_raw_flow_0p1s.csv', index=False)

        X_target, meta_df = flow_tensor.build_flow_tensor_windows(
            raw_flow_df,
            cameras=model_context['motion_cameras'],
            flow_sample_sec=model_context['flow_sample_sec'],
            window_sec=model_context['flow_tensor_window_sec'],
            hop_sec=model_context['flow_tensor_hop_sec'],
            grid=model_context['flow_grid'],
            feature_names=model_context['motion_feature_names'],
            broad_vib_score_config=model_context['flow_tensor_broad_vib_score_config'],
            reliable_soft_gate_config=model_context['flow_tensor_reliable_soft_gate_config'],
            accel_impact_score_config=model_context['flow_tensor_accel_impact_score_config'],
        )
        if len(X_target):
            X_target_model = X_target.astype(model_context['model_input_dtype'], copy=False)
            motion_raw, motion_scores = flow_tensor.score_flow_tensor_windows(X_target_model, model_context['motion_model'])
            motion_window_scores = meta_df.copy()
            motion_window_scores['category'] = sample.get('category', 'unknown')
            motion_window_scores['environment'] = sample.get('environment', 'unknown')
            motion_window_scores['target_group'] = target_group
            motion_window_scores['motion_anomaly_score_raw'] = motion_raw
            motion_window_scores['motion_anomaly_score'] = motion_scores
            motion_event_raw_scores = scoring.motion_event_raw_scores_from_tensor(
                X_target_model,
                model_context['motion_feature_names'],
            )
            motion_window_scores = scoring.add_motion_event_scores(
                motion_window_scores,
                raw_scores=motion_event_raw_scores,
                calibration=model_context['event_score_calibrations']['motion'],
                relative_config=model_context['motion_event_relative_score_config'],
            )
            motion_explanation_df = flow_tensor.explain_flow_tensor_windows(
                X_target_model,
                model_context['motion_model'],
                meta_df,
            )
            if len(motion_explanation_df):
                for col in motion_explanation_df.columns:
                    motion_window_scores[col] = motion_explanation_df[col].to_numpy()
            motion_scores_df = flow_tensor.aggregate_camera_scores(
                motion_window_scores,
                method=model_context['flow_tensor_score_aggregation'],
                score_col='motion_anomaly_score',
                raw_score_col='motion_anomaly_score_raw',
            )
            if len(motion_scores_df):
                motion_scores_df['target_group'] = target_group
        else:
            motion_scores_df = pd.DataFrame()

        audio_features_df = feature_pipeline.build_audio_features(sample, **model_context['audio_pipeline_kwargs'])
        if len(audio_features_df):
            audio_features_df = audio_features_df.copy()
            audio_features_df['target_group'] = target_group
            audio_raw, audio_scores = scoring.score_isolation_forest_artifact(audio_features_df, model_context['audio_model_artifact'])
            audio_scores_df = audio_features_df.copy()
            audio_scores_df['audio_anomaly_score_raw'] = audio_raw
            audio_scores_df['audio_anomaly_score'] = audio_scores
            audio_scores_df = scoring.add_audio_event_scores(
                audio_scores_df,
                model_context['event_score_calibrations']['audio'],
            )
        else:
            audio_scores_df = pd.DataFrame()

        scored_df = feature_pipeline.combine_audio_motion_scores(audio_scores_df, motion_scores_df, hop_sec=model_context['flow_tensor_hop_sec'])
        scored_df = scoring.add_composite_scores(scored_df, sync_score_config=model_context['sync_score_config'], final_score_weights=model_context['final_score_weights'])
        if len(scored_df):
            scored_df['target_group'] = target_group
            scored_df.to_csv(sample_output_dir / f'{safe_path_part(sample_id)}_window_scores.csv', index=False)
            scored_df.nlargest(TOP_N_ANOMALIES, 'anomaly_score').to_csv(sample_output_dir / f'{safe_path_part(sample_id)}_top_windows.csv', index=False)
            all_scored_parts.append(scored_df)
        if len(audio_features_df):
            audio_features_df.to_csv(sample_output_dir / f'{safe_path_part(sample_id)}_features.csv', index=False)
            all_feature_parts.append(audio_features_df)

    all_target_features_df = pd.concat(all_feature_parts, ignore_index=True) if all_feature_parts else pd.DataFrame()
    all_target_raw_flow_df = pd.concat(all_raw_flow_parts, ignore_index=True) if all_raw_flow_parts else pd.DataFrame()
    all_target_scored_df = pd.concat(all_scored_parts, ignore_index=True) if all_scored_parts else pd.DataFrame()
    cache_manifest_df = pd.DataFrame(cache_rows)
    audio_event_depth_manifest_df = write_audio_event_depth_outputs(
        target_samples=target_samples,
        all_target_scored_df=all_target_scored_df,
        model_context=model_context,
        target_output_root=target_output_root,
        comparison_dir=comparison_dir,
    )

    all_target_features_df.to_csv(comparison_dir / 'all_target_features.csv', index=False)
    all_target_raw_flow_df.to_csv(comparison_dir / 'all_target_raw_flow_0p1s.csv', index=False)
    all_target_scored_df.to_csv(comparison_dir / 'all_target_window_scores.csv', index=False)
    cache_manifest_df.to_csv(comparison_dir / 'flow_cache_manifest.csv', index=False)

    summaries = summarize_scored_outputs(all_target_scored_df, model_context)
    summaries['target_video_summary'].to_csv(comparison_dir / 'target_video_scores.csv', index=False)
    summaries['inference_result_table'].to_csv(comparison_dir / 'target_inference_result_table.csv', index=False)
    summaries['target_top_windows'].to_csv(comparison_dir / 'target_top_windows.csv', index=False)
    summaries['all_target_key_metrics_df'].to_csv(comparison_dir / 'all_target_key_metrics.csv', index=False)
    (comparison_dir / 'inference_cache_metadata.json').write_text(json.dumps(expected_cache_metadata, ensure_ascii=False, indent=2), encoding='utf-8')
    if len(summaries['target_video_summary']):
        for sample_id in target_samples['sample_id'].astype(str).tolist():
            sample_output_dir = target_output_root / safe_path_part(sample_id)
            sample_output_dir.mkdir(parents=True, exist_ok=True)
            summaries['target_video_summary'][summaries['target_video_summary']['video_id'].astype(str).eq(str(sample_id))].to_csv(sample_output_dir / f'{safe_path_part(sample_id)}_video_score.csv', index=False)

    outputs = {
        'all_target_features_df': all_target_features_df,
        'all_target_raw_flow_df': all_target_raw_flow_df,
        'all_target_scored_df': all_target_scored_df,
        'cache_manifest_df': cache_manifest_df,
        'audio_event_depth_manifest_df': audio_event_depth_manifest_df,
        **summaries,
    }
    print(f'saved comparison outputs: {comparison_dir}')
    return outputs


def concat_output_frames(outputs_list: list[dict[str, pd.DataFrame]], key: str) -> pd.DataFrame:
    parts = [outputs.get(key, pd.DataFrame()) for outputs in outputs_list]
    parts = [part for part in parts if part is not None and not part.empty]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def combine_score_outputs(outputs_list: list[dict[str, pd.DataFrame]]) -> dict[str, pd.DataFrame]:
    all_target_scored_df = concat_output_frames(outputs_list, 'all_target_scored_df')
    target_video_summary = concat_output_frames(outputs_list, 'target_video_summary')
    if len(target_video_summary) and 'max_anomaly_score' in target_video_summary.columns:
        target_video_summary = target_video_summary.sort_values('max_anomaly_score', ascending=False).reset_index(drop=True)
    target_top_windows = concat_output_frames(outputs_list, 'target_top_windows')
    all_target_key_metrics_df = concat_output_frames(outputs_list, 'all_target_key_metrics_df')
    cache_manifest_df = concat_output_frames(outputs_list, 'cache_manifest_df')
    return {
        'all_target_scored_df': all_target_scored_df,
        'target_video_summary': target_video_summary,
        'inference_result_table': build_inference_result_table(target_video_summary),
        'target_top_windows': target_top_windows,
        'all_target_key_metrics_df': all_target_key_metrics_df,
        'cache_manifest_df': cache_manifest_df,
    }


def pair_label_for(train_group: str, target_group: str) -> str:
    if len(str(train_group)) == 1 and len(str(target_group)) == 1:
        return f'{train_group}{target_group}'
    return f'{train_group}_{target_group}'


def write_pair_result_files(
    *,
    pair_dir: Path,
    pair_label: str,
    train_group: str,
    target_group: str,
    model_path: Path,
    normal_count: int,
    anomaly_count: int,
    outputs: dict[str, pd.DataFrame],
) -> Path:
    pair_dir.mkdir(parents=True, exist_ok=True)
    outputs['target_video_summary'].to_csv(pair_dir / f'target_video_scores_{pair_label}.csv', index=False)
    outputs['inference_result_table'].to_csv(pair_dir / f'target_inference_result_table_{pair_label}.csv', index=False)
    outputs['target_top_windows'].to_csv(pair_dir / f'target_top_windows_{pair_label}.csv', index=False)
    outputs['all_target_key_metrics_df'].to_csv(pair_dir / f'all_target_key_metrics_{pair_label}.csv', index=False)
    if 'all_target_scored_df' in outputs:
        outputs['all_target_scored_df'].to_csv(pair_dir / f'all_target_window_scores_{pair_label}.csv', index=False)
    return split_validation.write_inference_markdown_report(
        output_path=pair_dir / f'inference_result_{pair_label}.md',
        pair_label=pair_label,
        train_group=train_group,
        target_group=target_group,
        model_path=model_path,
        normal_count=normal_count,
        anomaly_count=anomaly_count,
        inference_result_table=outputs['inference_result_table'],
        video_summary=outputs['target_video_summary'],
        top_windows=outputs['target_top_windows'],
    )


In [ ]:
TARGET_INVENTORY_DF = load_movie_preprocess_inventory()
display(TARGET_INVENTORY_DF.groupby(['category', 'environment'], dropna=False).size().reset_index(name='count'))

if INFERENCE_RUN_MODE == 'single':
    TARGET_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    TARGET_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
    model_context = load_model_context(MODEL_PATH, FEATURE_CACHE_DIR)
    train_sample_ids = load_train_sample_ids(artifact_train_sample_ids=model_context['train_sample_ids'])
    target_samples = select_single_target_samples(TARGET_INVENTORY_DF, train_sample_ids)
    selected_targets_df = target_samples.copy()
    display(selected_targets_df[[c for c in ['sample_id', 'category', 'environment', 'front_path', 'rear_path', 'audio_path', 'front_frame_map_path', 'rear_frame_map_path'] if c in selected_targets_df.columns]])

    inference_outputs = score_target_samples(
        target_samples,
        model_context=model_context,
        target_output_root=TARGET_OUTPUT_ROOT,
        comparison_dir=TARGET_COMPARISON_DIR,
        reuse_cache=False,
    )
    if len(inference_outputs['inference_result_table']):
        display(inference_outputs['inference_result_table'])
        display(inference_outputs['target_video_summary'])
        display(inference_outputs['target_top_windows'])
        display(inference_outputs['all_target_key_metrics_df'].head(12))
    else:
        print('No scored windows were produced.')
    print('Graph output skipped; inference results are displayed as tables above.')
elif INFERENCE_RUN_MODE == 'split_validation':
    split_manifest_df = load_split_manifest(SPLIT_MANIFEST_PATH)
    group_labels = split_group_order(split_manifest_df)
    if len(group_labels) < 2:
        raise ValueError('split_validation inference requires at least two groups')
    display(split_manifest_df.groupby(['split_group', 'split_group_index', 'environment'], dropna=False).size().reset_index(name='count'))

    normal_groups: dict[str, pd.DataFrame] = {}
    for group_label in group_labels:
        group_df = split_manifest_df[split_manifest_df['split_group'].astype(str).eq(str(group_label))].reset_index(drop=True).copy()
        group_df['target_group'] = str(group_label)
        normal_groups[str(group_label)] = group_df

    anomaly_samples = TARGET_INVENTORY_DF[TARGET_INVENTORY_DF['category'].astype(str).eq('anomaly')].reset_index(drop=True).copy()
    if anomaly_samples.empty:
        raise ValueError('No anomaly samples were found in movie_preprocess inventory')
    anomaly_samples['target_group'] = 'anomaly'

    model_context_cache: dict[str, dict[str, Any]] = {}
    score_cache: dict[tuple[str, str], dict[str, pd.DataFrame]] = {}

    def context_for_group(train_group: str) -> dict[str, Any]:
        train_group = str(train_group)
        if train_group not in model_context_cache:
            model_path = SPLIT_OUTPUT_DIR / 'models' / safe_path_part(train_group) / 'isolation_forest_feature_poc.joblib'
            model_context_cache[train_group] = load_model_context(model_path, SPLIT_FEATURE_CACHE_DIR)
        return model_context_cache[train_group]

    def scored_outputs_for(train_group: str, target_set_name: str, samples: pd.DataFrame) -> dict[str, pd.DataFrame]:
        key = (str(train_group), str(target_set_name))
        if key not in score_cache:
            context = context_for_group(train_group)
            target_set_dir = SPLIT_OUTPUT_DIR / 'inference_cache' / f'model_{safe_path_part(train_group)}' / safe_path_part(target_set_name)
            score_cache[key] = score_target_samples(
                samples,
                model_context=context,
                target_output_root=target_set_dir,
                comparison_dir=target_set_dir / '_comparison',
                reuse_cache=REUSE_SPLIT_INFERENCE_CACHE,
            )
        return score_cache[key]

    pair_summary_rows: list[dict[str, Any]] = []
    for train_group, target_group in split_validation.ordered_group_pairs(group_labels):
        pair_label = pair_label_for(train_group, target_group)
        normal_outputs = scored_outputs_for(train_group, f'normal_{target_group}', normal_groups[str(target_group)])
        anomaly_outputs = scored_outputs_for(train_group, 'anomaly', anomaly_samples)
        combined_outputs = combine_score_outputs([normal_outputs, anomaly_outputs])
        pair_dir = SPLIT_OUTPUT_DIR / 'pair_results' / safe_path_part(pair_label)
        model_path = SPLIT_OUTPUT_DIR / 'models' / safe_path_part(train_group) / 'isolation_forest_feature_poc.joblib'
        markdown_path = write_pair_result_files(
            pair_dir=pair_dir,
            pair_label=pair_label,
            train_group=str(train_group),
            target_group=str(target_group),
            model_path=model_path,
            normal_count=len(normal_groups[str(target_group)]),
            anomaly_count=len(anomaly_samples),
            outputs=combined_outputs,
        )
        pair_summary_rows.append({
            'pair': pair_label,
            'train_group': str(train_group),
            'target_group': str(target_group),
            'normal_count': int(len(normal_groups[str(target_group)])),
            'anomaly_count': int(len(anomaly_samples)),
            'markdown_path': str(markdown_path),
            'pair_dir': str(pair_dir),
        })

    pair_summary_df = pd.DataFrame(pair_summary_rows)
    pair_summary_path = SPLIT_OUTPUT_DIR / 'pair_results' / 'pair_summary.csv'
    pair_summary_path.parent.mkdir(parents=True, exist_ok=True)
    pair_summary_df.to_csv(pair_summary_path, index=False)
    display(pair_summary_df)
    print(f'saved pair summary: {pair_summary_path}')
    print(f'inference cache entries executed or reused: {len(score_cache)}')
else:
    raise ValueError("INFERENCE_RUN_MODE must be 'single' or 'split_validation'")
